In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df=pd.read_csv( r'C:\Users\dell\Desktop\data analyst\SQL+PBI Projects\Lending Club\Data/accepted_2007_to_2018Q4.csv', low_memory=False)

In [3]:
print(df.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

In [4]:
print(df.shape) 
print(len(df.columns))  

(2260701, 151)
151


In [5]:
pd.set_option('display.max_rows', None)

nulls = df.isna().sum()
null_df = pd.DataFrame({
    'null_count': nulls,
    'null_pct': (nulls / len(df) * 100).round(2)
}).sort_values('null_pct', ascending=False)

print("Columns with nulls:", (nulls > 0).sum(), "out of", df.shape[1])
print("Columns with more than 50% nulls:", (null_df['null_pct'] > 50).sum())

null_df

Columns with nulls: 150 out of 151
Columns with more than 50% nulls: 44


,null_count,null_pct
member_id,2260701,100.00
orig_projected_additional_accrued_interest,2252050,99.62
hardship_end_date,2249784,99.52
hardship_start_date,2249784,99.52
hardship_type,2249784,99.52
hardship_reason,2249784,99.52
hardship_status,2249784,99.52
deferral_term,2249784,99.52
hardship_last_payment_amount,2249784,99.52
hardship_payoff_balance_amount,2249784,99.52


In [6]:
null_pct = df.isna().mean() * 100
cols_to_drop = null_pct[null_pct >= 50].index.tolist()
df = df.drop(columns=cols_to_drop)
print("Dropped columns:", len(cols_to_drop))
print("Remaining columns:", df.shape[1])

Dropped columns: 44
Remaining columns: 107


In [7]:
pd.set_option('display.max_rows', None)

null_info = pd.DataFrame({
    'column': df.columns,
    'null_pct': (df.isna().mean() * 100).round(2).values,
    'dtype': df.dtypes.astype(str).values
}).sort_values('null_pct', ascending=False).reset_index(drop=True)

null_info[null_info['null_pct'] > 0]

,column,null_pct,dtype
0,il_util,47.28,float64
1,mths_since_rcnt_il,40.25,float64
2,all_util,38.32,float64
3,open_rv_12m,38.31,float64
4,total_bal_il,38.31,float64
5,open_il_24m,38.31,float64
6,open_il_12m,38.31,float64
7,open_act_il,38.31,float64
8,open_acc_6m,38.31,float64
9,open_rv_24m,38.31,float64


## Section 1: Loan Terms and Borrower Info

In [8]:
mask = df['loan_amnt'].isna()
print(mask.sum())
print(df[mask]['loan_status'].isna().mean())

33
1.0


In [9]:
df = df.dropna(subset=['loan_amnt'])
print("Rows after dropping:", df.shape[0])

check = df[['emp_title', 'emp_length']]
print(pd.DataFrame({
    'null_count': check.isna().sum(),
    'null_pct': (check.isna().mean() * 100).round(2)
}))

Rows after dropping: 2260668
            null_count  null_pct
emp_title       166969      7.39
emp_length      146907      6.50


In [10]:
both = (df['emp_title'].isna() & df['emp_length'].isna()).sum()
only_title = (df['emp_title'].isna() & df['emp_length'].notna()).sum()
only_length = (df['emp_title'].notna() & df['emp_length'].isna()).sum()

print("Both null:", both)
print("emp_title null only:", only_title)
print("emp_length null only:", only_length)
print("Share of emp_length nulls that also have null emp_title:", round(both / df['emp_length'].isna().sum() * 100, 2), "%")

Both null: 146109
emp_title null only: 20860
emp_length null only: 798
Share of emp_length nulls that also have null emp_title: 99.46 %


In [11]:
df['emp_title'] = df['emp_title'].fillna('Unknown')
print("Remaining nulls in emp_title:", df['emp_title'].isna().sum())

Remaining nulls in emp_title: 0


In [12]:
section1 = ['id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment',
            'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
            'verification_status', 'issue_d', 'loan_status']

print("Columns in section 1:", len(section1))

pd.DataFrame({
    'null_count': df[section1].isna().sum(),
    'null_pct': (df[section1].isna().mean() * 100).round(4),
    'dtype': df[section1].dtypes.astype(str)
}).sort_values('null_count', ascending=False)

Columns in section 1: 16


,null_count,null_pct,dtype
emp_length,146907,6.4984,object
annual_inc,4,0.0002,float64
id,0,0.0000,object
loan_amnt,0,0.0000,float64
funded_amnt,0,0.0000,float64
funded_amnt_inv,0,0.0000,float64
term,0,0.0000,object
int_rate,0,0.0000,float64
installment,0,0.0000,float64
grade,0,0.0000,object


## Section 2: Borrower Credit Profile

In [13]:
section2 = ['pymnt_plan', 'url', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs',
            'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc',
            'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status']

print("Columns in section 2:", len(section2))

pd.DataFrame({
    'null_count': df[section2].isna().sum(),
    'null_pct': (df[section2].isna().mean() * 100).round(4),
    'dtype': df[section2].dtypes.astype(str)
}).sort_values('null_count', ascending=False)

Columns in section 2: 18


,null_count,null_pct,dtype
title,23326,1.0318,object
revol_util,1802,0.0797,float64
dti,1711,0.0757,float64
inq_last_6mths,30,0.0013,float64
open_acc,29,0.0013,float64
total_acc,29,0.0013,float64
delinq_2yrs,29,0.0013,float64
earliest_cr_line,29,0.0013,object
pub_rec,29,0.0013,float64
zip_code,1,0.0000,object


In [14]:
cols29 = ['delinq_2yrs', 'earliest_cr_line', 'open_acc', 'pub_rec', 'total_acc']
print(df[cols29].isna().all(axis=1).sum(), "rows null in all five")

29 rows null in all five


## Section 3: Loan Payment History

In [15]:
section3 = ['out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
            'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
            'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d', 'last_fico_range_high',
            'last_fico_range_low']

pd.DataFrame({
    'null_count': df[section3].isna().sum(),
    'null_pct': (df[section3].isna().mean() * 100).round(4),
    'dtype': df[section3].dtypes.astype(str)
}).sort_values('null_count', ascending=False)

,null_count,null_pct,dtype
last_pymnt_d,2427,0.1074,object
last_credit_pull_d,72,0.0032,object
out_prncp,0,0.0000,float64
out_prncp_inv,0,0.0000,float64
total_pymnt,0,0.0000,float64
total_pymnt_inv,0,0.0000,float64
total_rec_prncp,0,0.0000,float64
total_rec_int,0,0.0000,float64
total_rec_late_fee,0,0.0000,float64
recoveries,0,0.0000,float64


In [16]:
mask = df['last_pymnt_d'].isna()

print("Rows with null last_pymnt_d:", mask.sum())
print(df.loc[mask, 'loan_status'].value_counts())
print(df.loc[mask, 'total_pymnt'].describe())

Rows with null last_pymnt_d: 2427
loan_status
Charged Off                                            2313
Late (31-120 days)                                      102
Does not meet the credit policy. Status:Charged Off      12
Name: count, dtype: int64
count     2427.000000
mean      1366.360972
std       2430.652680
min          0.000000
25%          0.000000
50%        501.170000
75%       1840.015000
max      31584.000000
Name: total_pymnt, dtype: float64


In [17]:
mask = df['last_pymnt_d'].isna()
print(df.loc[mask, 'last_pymnt_amnt'].describe())
print("Rows with amount above zero:", (df.loc[mask, 'last_pymnt_amnt'] > 0).sum())

count    2427.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: last_pymnt_amnt, dtype: float64
Rows with amount above zero: 0


In [18]:
mask = df['last_pymnt_d'].isna()
df.loc[mask, ['total_rec_prncp', 'total_rec_int', 'recoveries', 'total_pymnt']].describe()

,total_rec_prncp,total_rec_int,recoveries,total_pymnt
count,2427.000000,2427.000000,2427.000000,2427.000000
mean,0.141801,0.125517,1366.093655,1366.360972
std,6.985748,6.183549,2430.767303,2430.652680
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,500.280000,501.170000
75%,0.000000,0.000000,1840.015000,1840.015000
max,344.150000,304.630000,31584.000000,31584.000000


## Section 4: Credit Warning Flags and Loan Setup

In [19]:
section4 = ['acc_now_delinq', 'delinq_amnt', 'collections_12_mths_ex_med', 'chargeoff_within_12_mths',
            'pub_rec_bankruptcies', 'tax_liens', 'application_type', 'policy_code',
            'disbursement_method', 'hardship_flag', 'debt_settlement_flag']

In [20]:
null_info = pd.DataFrame({
    'null_count': df[section4].isna().sum(),
    'null_pct': (df[section4].isna().mean() * 100).round(4),
    'dtype': df[section4].dtypes.astype(str)
}).sort_values('null_count', ascending=False)

print(null_info)
print(df['policy_code'].value_counts())

                            null_count  null_pct    dtype
pub_rec_bankruptcies              1365    0.0604  float64
collections_12_mths_ex_med         145    0.0064  float64
chargeoff_within_12_mths           145    0.0064  float64
tax_liens                          105    0.0046  float64
acc_now_delinq                      29    0.0013  float64
delinq_amnt                         29    0.0013  float64
application_type                     0    0.0000   object
policy_code                          0    0.0000  float64
disbursement_method                  0    0.0000   object
hardship_flag                        0    0.0000   object
debt_settlement_flag                 0    0.0000   object
policy_code
1.0    2260668
Name: count, dtype: int64


## Section 5: Balances and Credit Limits

In [21]:
section5 = ['tot_cur_bal', 'avg_cur_bal', 'tot_hi_cred_lim', 'total_rev_hi_lim', 'total_bc_limit',
            'total_il_high_credit_limit', 'total_bal_ex_mort', 'total_bal_il', 'max_bal_bc',
            'bc_open_to_buy', 'tot_coll_amt']

null_info = pd.DataFrame({
    'null_count': df[section5].isna().sum(),
    'null_pct': (df[section5].isna().mean() * 100).round(4),
    'dtype': df[section5].dtypes.astype(str)
}).sort_values('null_count', ascending=False)

print(null_info)

year = df['issue_d'].str[-4:]
print(df[['total_bal_il', 'max_bal_bc']].isna().groupby(year).mean().mul(100).round(1))

                            null_count  null_pct    dtype
total_bal_il                    866129   38.3130  float64
max_bal_bc                      866129   38.3130  float64
bc_open_to_buy                   74935    3.3147  float64
avg_cur_bal                      70346    3.1117  float64
tot_cur_bal                      70276    3.1086  float64
tot_hi_cred_lim                  70276    3.1086  float64
total_rev_hi_lim                 70276    3.1086  float64
total_il_high_credit_limit       70276    3.1086  float64
tot_coll_amt                     70276    3.1086  float64
total_bc_limit                   50030    2.2131  float64
total_bal_ex_mort                50030    2.2131  float64
         total_bal_il  max_bal_bc
issue_d                          
2007            100.0       100.0
2008            100.0       100.0
2009            100.0       100.0
2010            100.0       100.0
2011            100.0       100.0
2012            100.0       100.0
2013            100.0       100.

In [22]:
dash_cols = ['id', 'issue_d', 'loan_amnt', 'funded_amnt', 'int_rate', 'term', 'loan_status',
             'grade', 'sub_grade', 'total_pymnt', 'total_rec_int', 'recoveries',
             'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'revol_util', 'emp_length',
             'home_ownership', 'verification_status', 'delinq_2yrs', 'application_type',
             'purpose', 'addr_state']

print("Columns:", len(dash_cols))

pd.DataFrame({
    'null_count': df[dash_cols].isna().sum(),
    'null_pct': (df[dash_cols].isna().mean() * 100).round(4),
    'dtype': df[dash_cols].dtypes.astype(str)
}).sort_values('null_count', ascending=False)

Columns: 24


,null_count,null_pct,dtype
emp_length,146907,6.4984,object
revol_util,1802,0.0797,float64
dti,1711,0.0757,float64
delinq_2yrs,29,0.0013,float64
annual_inc,4,0.0002,float64
issue_d,0,0.0000,object
purpose,0,0.0000,object
application_type,0,0.0000,object
verification_status,0,0.0000,object
home_ownership,0,0.0000,object


Columns to PowerBI

In [27]:
import os

dash_cols =['id', 'issue_d', 'loan_amnt', 'funded_amnt', 'int_rate', 'term', 'loan_status',
              'grade', 'sub_grade', 'total_pymnt', 'total_rec_int', 'total_rec_prncp', 'recoveries',
             'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'revol_util', 'emp_length',
             'home_ownership', 'verification_status', 'delinq_2yrs', 'application_type',
             'purpose', 'addr_state','collection_recovery_fee','out_prncp']

out_path = r'C:\Users\dell\Desktop\data analyst\SQL+PBI Projects\Lending Club\Data\loans_dashboard.csv'

df_dash = df[dash_cols].copy()
print("Columns:", len(dash_cols))
print("Shape:", df_dash.shape)

print(pd.DataFrame({
    'null_count': df_dash.isna().sum(),
    'null_pct': (df_dash.isna().mean() * 100).round(4),
    'dtype': df_dash.dtypes.astype(str)
}).sort_values('null_count', ascending=False))

df_dash.to_csv(out_path, index=False)
print("Saved:", out_path)
print("Size (MB):", round(os.path.getsize(out_path) / 1024 / 1024, 1))

Columns: 27
Shape: (2260668, 27)
                         null_count  null_pct    dtype
emp_length                   146907    6.4984   object
revol_util                     1802    0.0797  float64
dti                            1711    0.0757  float64
delinq_2yrs                      29    0.0013  float64
annual_inc                        4    0.0002  float64
collection_recovery_fee           0    0.0000  float64
addr_state                        0    0.0000   object
purpose                           0    0.0000   object
application_type                  0    0.0000   object
verification_status               0    0.0000   object
home_ownership                    0    0.0000   object
fico_range_high                   0    0.0000  float64
fico_range_low                    0    0.0000  float64
id                                0    0.0000   object
issue_d                           0    0.0000   object
recoveries                        0    0.0000  float64
total_rec_prncp                 

In [28]:
print("Shape:", df_dash.shape)

Shape: (2260668, 27)
